In [1]:
import pandas as pd

import pickle
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

import _helper_grammar_functions as jsgo

from _helper_functions import loadAff
from _helper_functions import get_kmers
from _helper_functions import zipdf

import _syntax_score_helper_function as ga

pd.set_option('display.max_colwidth', 100)


In [2]:

# load syntax featureess 
fn='../../0-orl/1-mpra-experiment/F05-Syntax-Features/6-real-and-imputed-motifs.tsv'
mDF=pd.read_csv(fn,sep='\t').set_index('phrase-key')
    
sigMotifSet=set(mDF.index)

mDF.head(2)

,phrase-length,func-odds-ratio,real-data,hyp-func-p-bonf,num-total-seqs,num-active-ens,obs-or-imputed
phrase-key,,,,,,,
ord_ori_spc=g_21_e_24_E,3.0,11.206427,"(63, 34, 7)",3.246130e-29,104.0,63.0,obs
ord_ori_spc=g_13_e_24_E,3.0,8.385350,"(108, 73, 21)",3.208155e-43,202.0,108.0,obs


# Encoding enhancer feature vectors for these models

In [3]:
# get activity of all variants

fn=f'01-Barcode-Processing/endf_final-experiment-enhancer-activity.pd.df.pickle'
endf=pd.read_pickle(fn).set_index('enhancer-seq',drop=False)
endf.head(1)

,enhancer-seq,enhancer-id,barcode-seq,rpm-D1,rpm-D2,rpm-D3,rpm-D4,rpm-R1,rpm-R2,rpm-R3,...,ratio-repagg-5-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean,ratio-replist-6-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5,ratio-repagg-6-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean,ratio-replist-7-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5,ratio-repagg-7-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum-log2,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum-log2-rescaled-minpctl55-maxpctl65
enhancer-seq,,,,,,,,,,,,,,,,,,,,,
AAAAAACTTGAAATCGCGTCTTATGCAACTGTAAATCCGGGAGCAGATAAATACACAAATATCCTG,AAAAAACTTGAAATCGCGTCTTATGCAACTGTAAATCCGGGAGCAGATAAATACACAAATATCCTG,Chr8:6147080-6147146,"[TTGCTAACACGATTGCGGATCGATT, ACTTATCTACTGACTGTCCTCGCCT, TTTTGTATCAAGCTTGTGGGACCCG, GCTATGTTATCTGC...","[0.220733824144946, 0.23371713495616817, 0.15564169227966881, 0.4924309922433955, 0.038875310318...","[0.28989230073817873, 0.24411855465032495, 0.15240996659371667, 0.3197764512875319, 0.0304542638...","[0.2338136718273605, 0.37410018092653813, 0.14791915315194892, 0.35002404587412034, 0.0855597031...","[0.31803960026920236, 0.3066798448131834, 0.12480684722132872, 0.24939225940386225, 0.0906861328...","[nan, nan, nan, nan, nan, nan, 5.203892531585922, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]",...,2.358607,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 33.78846761552279, 0.0, 0.0, 0.0]",3.754274,"[2.2921863664721522, 0.0, 0.0, 0.0, 2.358606678717607, 0.0, 3.7542741795025325]","[2.2921863664721522, nan, nan, nan, 2.358606678717607, nan, 3.7542741795025325]",8.405067,3.071259,-3.625061


In [4]:
len(endf)

3318

# Only count syntax scores for ETS and GATA higher than .1 and .2 affinities, respectively

In [5]:
gata2aff=loadAff('ref/parsed_Gata6_3769_contig8mers.txt')
ets2aff =loadAff('ref/parsed_Ets1_8mers.txt')

In [6]:

etsCores=set(['GGAA','GGAT','TTCC','ATCC'])
gataCores=set(['GATA','TATC'])

allGataList=[]
allEtsList =[]

for seq in endf['enhancer-seq']:
        
    gatalist=[]
    etslist=[]
    for i,kmer in enumerate(get_kmers(seq,8)):
        
        core=kmer[2:6]
        if core in gataCores:
            # print(kmer)
            gatalist.append((i,gata2aff[kmer]))
            
        if core in etsCores:
            etslist.append((i,ets2aff[kmer]))
            
    allGataList.append(gatalist)
    allEtsList .append(etslist)
    
endf['gata-affs']=allGataList
endf['ets-affs'] =allEtsList

In [7]:
endf.head(1)

,enhancer-seq,enhancer-id,barcode-seq,rpm-D1,rpm-D2,rpm-D3,rpm-D4,rpm-R1,rpm-R2,rpm-R3,...,ratio-repagg-6-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean,ratio-replist-7-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5,ratio-repagg-7-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum-log2,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum-log2-rescaled-minpctl55-maxpctl65,gata-affs,ets-affs
enhancer-seq,,,,,,,,,,,,,,,,,,,,,
AAAAAACTTGAAATCGCGTCTTATGCAACTGTAAATCCGGGAGCAGATAAATACACAAATATCCTG,AAAAAACTTGAAATCGCGTCTTATGCAACTGTAAATCCGGGAGCAGATAAATACACAAATATCCTG,Chr8:6147080-6147146,"[TTGCTAACACGATTGCGGATCGATT, ACTTATCTACTGACTGTCCTCGCCT, TTTTGTATCAAGCTTGTGGGACCCG, GCTATGTTATCTGC...","[0.220733824144946, 0.23371713495616817, 0.15564169227966881, 0.4924309922433955, 0.038875310318...","[0.28989230073817873, 0.24411855465032495, 0.15240996659371667, 0.3197764512875319, 0.0304542638...","[0.2338136718273605, 0.37410018092653813, 0.14791915315194892, 0.35002404587412034, 0.0855597031...","[0.31803960026920236, 0.3066798448131834, 0.12480684722132872, 0.24939225940386225, 0.0906861328...","[nan, nan, nan, nan, nan, nan, 5.203892531585922, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]",...,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 33.78846761552279, 0.0, 0.0, 0.0]",3.754274,"[2.2921863664721522, 0.0, 0.0, 0.0, 2.358606678717607, 0.0, 3.7542741795025325]","[2.2921863664721522, nan, nan, nan, 2.358606678717607, nan, 3.7542741795025325]",8.405067,3.071259,-3.625061,"[(43, 0.6840717119380976), (57, 0.06838836517828153)]","[(32, 0.2500506805527969), (58, 0.3887981644425242)]"


# only consider elements with non overlapping sites

In [8]:
def get_tf_olap_info(gsites_beforeAffFilter,esites_beforeAffFilter,tf2minaff):
    
    startIndexesCallable=set()
    gsites=[] # gata sites with callable aff
    esites=[] # gata sites with callable aff
    
    for tfname,tflist in [('G',gsites_beforeAffFilter),('E',esites_beforeAffFilter)]:
        
        minAff=tf2minaff[tfname]
        for tfstart,tfaff in tflist:
            
            if tfaff>=minAff:
                if   tfname=='G':gsites.append((tfstart,tfaff))
                elif tfname=='E':esites.append((tfstart,tfaff))
                startIndexesCallable.add(tfstart)
                

    # get original tf position mapped to seq idx
    allsiteIdxs_beforeAffFilter=gsites_beforeAffFilter+esites_beforeAffFilter
    allsiteIdxs_beforeAffFilter=sorted([idx for idx,aff in allsiteIdxs_beforeAffFilter])
    startidx2originaltfpos={}
    for tfpos,seqidx in enumerate(allsiteIdxs_beforeAffFilter):
        startidx2originaltfpos[seqidx]=tfpos
    
    # now that you have callable sites, determine which are not usable due to having overlaps
    
    allsites=gsites+esites                            # all sites with callable aff
    allsiteIdxs=sorted([idx for idx,aff in allsites]) # gata sites with callable aff
    
    # get overlap info for sufficiently high aff sites
    tfstart2seqidxlocs={}
    seqidx2occupants={}
    for idx in allsiteIdxs:
        
        for bpOccupied in range(idx,idx+8):
            
            if idx not in tfstart2seqidxlocs:
                tfstart2seqidxlocs[idx]=set()
            tfstart2seqidxlocs[idx].add(bpOccupied)
            
            if bpOccupied not in seqidx2occupants:
                seqidx2occupants[bpOccupied]=0
            seqidx2occupants[bpOccupied]+=1
            
    # determine site overlap
    tf2sitetype2count={}
    tf2sitetype2starts={}
    distinctSiteSet=set()
    for tfname,tflist in [('G',gsites),('E',esites)]:
        
        tf2sitetype2starts[tfname]={'dist':set(),'olap':set()}
            
        tfstartlist=[start for start,aff in tflist]
        
        numDistinctSites=0
        numOlapSites=0
        for tfi in tfstartlist:

            isDistinct=True
            for seqidx in tfstart2seqidxlocs[tfi]:
                if seqidx2occupants[seqidx]>=2:
                    isDistinct=False

            if isDistinct: 
                numDistinctSites+=1
                tf2sitetype2starts[tfname]['dist'].add(tfi)
                distinctSiteSet.add(tfi)
                
            else:          
                numOlapSites+=1
                tf2sitetype2starts[tfname]['olap'].add(tfi)
            
        tf2sitetype2count[tfname]={'ndist':numDistinctSites,'nolap':numOlapSites}
    
    tfNumCallableList=[]
    tfNumDistinctList=[]
    for tfstart in allsiteIdxs:
        if tfstart in distinctSiteSet:      tfNumDistinctList.append(startidx2originaltfpos[tfstart])
        if tfstart in startIndexesCallable: tfNumCallableList.append(startidx2originaltfpos[tfstart])
    
    return tfNumCallableList,tf2sitetype2count,tf2sitetype2starts,tfNumDistinctList
    

In [11]:
# we define a 'called' site as >.2 gata, >.1 ets

# we want 2 'distinct' sites for gata and 2 for ets
# a distinct site is
    # non overlapping called site... or
    # a called site overlappingi a non-called site (eg, g<.2 or e<.1)

# 2 called sites overalpping are not distinct.

# any enhancers with ≥1 overlapping called sites are currently removed from the analysis

# enhancer types
#  notsuf  - not enough called sites (≥2e ≥2g)
#  olap    - sufficient number called sites, but ≥1 overlaps another called site
#  notolap - ≥2gata and ≥2ets called sites dont overlap

minGToBeSite=.2
minEToBeSite=.1
tf2minaff={'G':minGToBeSite,'E':minEToBeSite}

# min num distinct 
nminG=1
nminE=1
nminTot=3

# max num overlap
maxOverlap=1

numberSitesLabelList=[]

siteIdxCalledList        =[]
siteIdxCalledDistinctList=[]
numDistinctGataList      =[]
numDistinctEtsList       =[]
numDistinctTotList       =[]
numOlappingSitesList         =[]

for gsites,esites in zipdf(endf,['gata-affs','ets-affs']):
    
    # print(gsites,esites)
    tfNumCallableList,tf2sitetype2count,tf2sitetype2starts,tfNumDistinctList = get_tf_olap_info(gsites,esites,tf2minaff)
        
    numTotalSites_beforeAffFilter=len(gsites+esites)
    
    numDistGata=tf2sitetype2count['G']['ndist']
    numDistEts =tf2sitetype2count['E']['ndist']
    numDistTot =numDistGata+numDistEts
    
    numCallableTotal=len(tfNumCallableList)
    numOlapTotal=numCallableTotal-numDistTot
        
    siteIdxCalledList        .append(tfNumCallableList)
    siteIdxCalledDistinctList.append(tfNumDistinctList)
    numDistinctGataList      .append(numDistGata)
    numDistinctEtsList       .append(numDistEts )
    numDistinctTotList       .append(numDistTot )
    numOlappingSitesList         .append(numOlapTotal)
        
        
# endf['olap-annotation'] = 

        
endf['site-idx-list-callable']=siteIdxCalledList        
endf['site-idx-list-callable-distinct']=siteIdxCalledDistinctList
endf['num-gata-distinct']=numDistinctGataList      
endf['num-ets-distinct']=numDistinctEtsList       
endf['num-tot-distinct']=numDistinctTotList  
endf['num-tot-olap']=numOlappingSitesList  

In [12]:
endf.head(1)

,enhancer-seq,enhancer-id,barcode-seq,rpm-D1,rpm-D2,rpm-D3,rpm-D4,rpm-R1,rpm-R2,rpm-R3,...,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum-log2,ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum-log2-rescaled-minpctl55-maxpctl65,gata-affs,ets-affs,site-idx-list-callable,site-idx-list-callable-distinct,num-gata-distinct,num-ets-distinct,num-tot-distinct,num-tot-olap
enhancer-seq,,,,,,,,,,,,,,,,,,,,,
AAAAAACTTGAAATCGCGTCTTATGCAACTGTAAATCCGGGAGCAGATAAATACACAAATATCCTG,AAAAAACTTGAAATCGCGTCTTATGCAACTGTAAATCCGGGAGCAGATAAATACACAAATATCCTG,Chr8:6147080-6147146,"[TTGCTAACACGATTGCGGATCGATT, ACTTATCTACTGACTGTCCTCGCCT, TTTTGTATCAAGCTTGTGGGACCCG, GCTATGTTATCTGC...","[0.220733824144946, 0.23371713495616817, 0.15564169227966881, 0.4924309922433955, 0.038875310318...","[0.28989230073817873, 0.24411855465032495, 0.15240996659371667, 0.3197764512875319, 0.0304542638...","[0.2338136718273605, 0.37410018092653813, 0.14791915315194892, 0.35002404587412034, 0.0855597031...","[0.31803960026920236, 0.3066798448131834, 0.12480684722132872, 0.24939225940386225, 0.0906861328...","[nan, nan, nan, nan, nan, nan, 5.203892531585922, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]",...,3.071259,-3.625061,"[(43, 0.6840717119380976), (57, 0.06838836517828153)]","[(32, 0.2500506805527969), (58, 0.3887981644425242)]","[0, 1, 3]","[0, 1, 3]",1,2,3,0


In [13]:
activityColName='ratio-expagg-1234567-minDnaRpm0.05-filtNa-Dnotnull-minDnaBc5-mean-minActFilt1.5-minRepsActObs3-expsum-log2-rescaled-minpctl55-maxpctl65'
outcols=['enhancer-seq','enhancer-id',activityColName,'site-idx-list-callable','site-idx-list-callable-distinct','num-gata-distinct','num-ets-distinct','num-tot-distinct','num-tot-olap','gata-affs','ets-affs']

fn=f'01-Barcode-Processing/endf_final-experiment-enhancer-activity_usable-sites-annotated'
endf.loc[:,outcols].to_pickle(fn+'.pd.df.pickle')

# annotate syntax scores

In [14]:
beta=False

sitesToUseCol='site-idx-list-callable'

effectSize = 'odds-ratio'

# If writing


eDF=ga.get_all_grammar_motifs_BATCH(endf,
                                    sigMotifSet,
                                    mDF,
                                    f'func-{effectSize}',
                                    f'nonfunc-{effectSize}',
                                    sitesToUseCol=sitesToUseCol,
                                    skip_aff=True,
                                    activatorOnly=True)


eDF.to_pickle(f'02-Syntax-Scores/1i-all-ens-with-motifs-annotated.effectSize={effectSize}.pandas.dataframe.pickle')

# Collapse onto sequences
Col2Func={'fc':list}#,'n-enhancers':list,'motif-count':sum}

eDF=eDF.loc[:,['seq','type']+list(Col2Func.keys())].groupby(['seq','type']).agg(Col2Func).reset_index().set_index('seq')

eDF.columns=['Type','M']

aDF=eDF.loc[eDF.Type=='F',['M']]
aDF.columns=['Ma']

finalDF=pd.concat([aDF],axis=1)


# Sum syntax features

for colBase in ['M']:
    for Type in ['a']:
        col=colBase+Type

        newcol=col+'-sum'
        finalDF[newcol]=finalDF[col].apply(lambda l: sum(l) if type(l)==list else np.NaN)


finalDF.to_pickle(f'02-Syntax-Scores/2i-all-seqs-with-motif-info.effectSize={effectSize}.pandas.dataframe.pickle')
    
finalDF[f'Ma-sum-nafill1']=finalDF['Ma-sum'].fillna(1)


finalDF.to_pickle(f'02-Syntax-Scores/3i-all-seqs.fillNaN.effectSize={effectSize}.pandas.dataframe.pickle')
        
print('done :)')

done :)
